### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="pva_revenue_prediction_kddcup98",
    dataset_year="1997",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://kdd.ics.uci.edu/databases/kddcup98/kddcup98.html", # Rel 10.24432/C5401H
    download_description="""

wget https://kdd.ics.uci.edu/databases/kddcup98/epsilon_mirror/cup98lrn.zip && unzip cup98lrn && rm cup98lrn.zip
wget https://kdd.ics.uci.edu/databases/kddcup98/epsilon_mirror/cup98val.zip && unzip cup98val && rm cup98val.zip && wget https://kdd.ics.uci.edu/databases/kddcup98/epsilon_mirror/valtargt.txt
mkdir -p local-data-warehouse/pva_revenue_prediction_kddcup98 && mv cup98VAL.txt local-data-warehouse/pva_revenue_prediction_kddcup98/ && mv cup98LRN.txt local-data-warehouse/pva_revenue_prediction_kddcup98/ && mv valtargt.txt local-data-warehouse/pva_revenue_prediction_kddcup98/
""",
    # References
    academic_reference_bibtex=r"""@misc{Parsa1998KDDCup1998,
  author = {Ismail Parsa},
  title  = {KDD Cup 1998},
  year   = {1998},
  howpublished = {\url{https://kdd.ics.uci.edu/databases/kddcup98/kddcup98.html}},
  note   = {Dataset, UCI Machine Learning Repository. DOI: 10.24432/C5401H}
}
""",
    academic_reference_bibtex_key="Parsa1998KDDCup1998",
    license=None, # Unclear but has usage instruction that do not count as license (?)
    data_tags=["IID"],
    curation_comments="""
We start with the train and validation data from the KDD Cup 1998 website and combine them.

- The description of the task and data for the KDD Cup (https://kdd.ics.uci.edu/databases/kddcup98/epsilon_mirror/cup98doc.txt) is among the best real-world dataset and task descriptions I have ever seen.
- The task was made IID through feature engineering and would usually used to predict the next year. We only predict from the training data from the current year, other samples from the current year (as to see who else to contact in the same year). Features contain information from the previous year.
- The task is revenue generation and the score function is profit-based with a cost of $0.68 per prediction. The higher the revenue generated, the better. We use the binary classification target (TARGET_B) to predict if a person would donate.
- The TCODE contains titles now in the data dictionary. We map large amount of missing values to placeholder titles and the smaller ones to a new "other" title.
- We treat data as string or category based on the data dictionary suggestions. In cases where the column is from a predefined set of choices, it becomes categorical. We treat various free-text categorical as string columns (they are also high-cardinality otherwise).
- We transform the date columns to pandas datetime.
- The data contains spatial information (ZIP, STATE). We do not resolve them and leave them for the pipeline to handle.
- We remove a faulty postfix ("-") in the ZIP code.
- We drop rows with PVASTATE='E' as this represent a group of donors represent by a different organization chapter. These are 8 rows only.
- The data contains two entries with a faulty day of birth (born in month 00). We drop both rows, as we assume this is indicative of faulty data.
- We noticed that nan values and dtypes mismatch for NOEXCH do correlate with the target. We add these cases as their own categories.
- We resolve the ordinal encoding and nan encoding for all columns.
- Note, that data has a lot of features that contain data from a similar origin that was transformed into individual columns.
- We do not decode the promotion codes across years but keep them as string columns as it would blow up the dimensionality too much otherwise.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="TARGET_B",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="TARGET_B",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "cup98LRN.txt")
val_df = pd.read_csv(dataset_mold.path / "cup98VAL.txt")
val_targets = pd.read_csv(dataset_mold.path / "valtargt.txt")
df = pd.concat([df, val_df.merge(val_targets, on="CONTROLN")], axis=0).reset_index(drop=True)

/tmp/ipykernel_119432/1021879402.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_mold.path / "cup98LRN.txt")
/tmp/ipykernel_119432/1021879402.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  val_df = pd.read_csv(dataset_mold.path / "cup98VAL.txt")


In [3]:
title_code_map = {0: "No Title", 1: "MR.", 1001: "MESSRS.", 1002: "MR. & MRS.", 2: "MRS.", 2002: "MESDAMES", 3: "MISS", 3003: "MISSES", 4: "DR.", 4002: "DR. & MRS.", 4004: "DOCTORS", 5: "MADAME", 6: "SERGEANT", 9: "RABBI", 10: "PROFESSOR", 10002: "PROFESSOR & MRS.", 10010: "PROFESSORS", 11: "ADMIRAL", 11002: "ADMIRAL & MRS.", 12: "GENERAL", 12002: "GENERAL & MRS.", 13: "COLONEL", 13002: "COLONEL & MRS.", 14: "CAPTAIN", 14002: "CAPTAIN & MRS.", 15: "COMMANDER", 15002: "COMMANDER & MRS.", 16: "DEAN", 17: "JUDGE", 17002: "JUDGE & MRS.", 18: "MAJOR", 18002: "MAJOR & MRS.", 19: "SENATOR", 20: "GOVERNOR", 21002: "SERGEANT & MRS.", 22002: "COLNEL & MRS.", 24: "LIEUTENANT", 26: "MONSIGNOR", 27: "REVEREND", 28: "MS.", 28028: "MSS.", 29: "BISHOP", 31: "AMBASSADOR", 31002: "AMBASSADOR & MRS.", 33: "CANTOR", 36: "BROTHER", 37: "SIR", 38: "COMMODORE", 40: "FATHER", 42: "SISTER", 43: "PRESIDENT", 44: "MASTER", 46: "MOTHER", 47: "CHAPLAIN", 48: "CORPORAL", 50: "ELDER", 56: "MAYOR", 59002: "LIEUTENANT & MRS.", 62: "LORD", 63: "CARDINAL", 64: "FRIEND", 65: "FRIENDS", 68: "ARCHDEACON", 69: "CANON", 70: "BISHOP", 72002: "REVEREND & MRS.", 73: "PASTOR", 75: "ARCHBISHOP", 85: "SPECIALIST", 87: "PRIVATE", 89: "SEAMAN", 90: "AIRMAN", 91: "JUSTICE", 92: "MR. JUSTICE", 100: "M.", 103: "MLLE.", 104: "CHANCELLOR", 106: "REPRESENTATIVE", 107: "SECRETARY", 108: "LT. GOVERNOR", 109: "LIC.", 111: "SA.", 114: "DA.", 116: "SR.", 117: "SRA.", 118: "SRTA.", 120: "YOUR MAJESTY", 122: "HIS HIGHNESS", 123: "HER HIGHNESS", 124: "COUNT", 125: "LADY", 126: "PRINCE", 127: "PRINCESS", 128: "CHIEF", 129: "BARON", 130: "SHEIK", 131: "PRINCE AND PRINCESS", 132: "YOUR IMPERIAL MAJEST", 135: "M. ET MME.", 210: "PROF."}
recency_map = {
    "C": "Current Donor",
    "L": "Lapsed Donor",
    "I": "Inactive Donor",
    "D": "Dormant Donor",
}
frequency_map = {
    "1": "One gift in the period of recency",
    "2": "Two-Four gifts in the period of recency",
    "5": "Five+ gifts in the period of recency",
}
amount_map = {
    "L": "Less than $100 (Low Dollar)",
    "C": "$100-499 (Core)",
    "M": "$500-999 (Major)",
    "T": "$1,000+ (Top)",
}
urbanicity_map = {
    "U": "Urban",
    "C": "City",
    "S": "Suburban",
    "T": "Town",
    "R": "Rural",
}
ses_map = {
    "1": "Highest SES",
    "2": "Average SES",
    "3": "Lowest SES",
    "4": "Lower Lowest SES in Urban communities", # weird name to show it is lower as otherwise we would need to rename 2 and 3 only for urban cases, but this should be fine.
}

In [4]:
# Handle DOB
df = df[~df["DOB"].replace(np.nan, 1111).astype(float).astype(int).astype(str).str.zfill(4).str.endswith("00")] # drop 2 rows with month 00
df["DOB"] = df["DOB"].replace(0, np.nan)
dob_nan_mask = df["DOB"].isna()
df.loc[dob_nan_mask, "DOB"] = 0
df["DOB"] = df["DOB"].astype(float).astype(int).astype("string").str.zfill(4)
df.loc[dob_nan_mask, "DOB"] = np.nan
df["DOB"] = pd.to_datetime("19" + df["DOB"], format="%Y%m")

# Make pd Dates (unclear if best preprocessing is not just int as in original data but this way we have the correct metadata of the col)
prom_dates = [
    "ADATE_2","ADATE_3","ADATE_4","ADATE_5","ADATE_6","ADATE_7","ADATE_8",
    "ADATE_9","ADATE_10","ADATE_11","ADATE_12","ADATE_13","ADATE_14",
    "ADATE_15","ADATE_16","ADATE_17","ADATE_18","ADATE_19","ADATE_20",
    "ADATE_21","ADATE_22","ADATE_23","ADATE_24",
    "RDATE_3","RDATE_4","RDATE_5","RDATE_6","RDATE_7","RDATE_8","RDATE_9",
    "RDATE_10","RDATE_11","RDATE_12","RDATE_13","RDATE_14","RDATE_15",
    "RDATE_16","RDATE_17","RDATE_18","RDATE_19","RDATE_20","RDATE_21",
    "RDATE_22","RDATE_23","RDATE_24",
    "MAXADATE", "ODATEDW",
    "MINRDATE", "MAXRDATE", "LASTDATE", "FISTDATE", "NEXTDATE",
]
df["FISTDATE"] = df["FISTDATE"].replace(0, np.nan)
for col in prom_dates:
    nan_mask = df[col].isna()
    df.loc[nan_mask, col] = 1111
    df[col] = pd.to_datetime("19" + df[col].astype(int).astype("string"), format="%Y%m")
    df.loc[nan_mask, col] = np.nan

# Update title map
missing_title_in_data_dic = list(np.unique(df[~df["TCODE"].isin(title_code_map.keys())]["TCODE"]))
for code in missing_title_in_data_dic:
    title_code_map[code] = f"Missing Title {code}"
# Replace Title
assert df["TCODE"].isin(title_code_map.keys()).all(), "Some TCODE values are not in the title_code_map"
df["Title"] = df["TCODE"].map(title_code_map)
df = df.drop(columns=["TCODE"])

# Handle outsource codes
df["OSOURCE"] = df["OSOURCE"].replace(" ", np.nan)
# Remove postfix from ZIP
df["ZIP"] = df["ZIP"].str.replace("-", "")
# Handle mailcode
df["MAILCODE"] = df["MAILCODE"].replace({" ": "Address is OK", "B": "Bad Address"})
# Handle missing values PVASTATE
df["PVASTATE"] = df["PVASTATE"].replace(" ", np.nan)
df = df[df["PVASTATE"] != "E"]
# a scatterplot of these values and the targets shows some superficial correlation
df["NOEXCH"] = df["NOEXCH"].replace({" ": "can be exchanged", "X": "cannot be exchanged", 0: "0-nan-case", "0": "0-nan-case", 1: "1-nan-case", "1": "1-nan-case"})# .value_counts(dropna=False)
# -- Resolve some of the cat variables
df["RECINHSE"] = df["RECINHSE"].replace({
    " ": "Not an In House Record",
    "X": "Donor has given to PVA's In House program"
})
df["RECP3"] = df["RECP3"].replace({
    " ": "Not a P3 Record",
    "X": "Donor has given to PVA's P3 program"
})
df["RECPGVG"] = df["RECPGVG"].replace({
    " ": "Not a Planned Giving Record",
    "X": "Planned Giving Record"
})
df["RECSWEEP"] = df["RECSWEEP"].replace({
    " ": "Not a Sweepstakes Record",
    "X": "Sweepstakes Record"
})
# -- Resolve MDMAUD
# Split into 4 bytes (pad right so short strings don't error)
b = df["MDMAUD"].str.pad(4, side="right").str[:4]
b1, b2, b3 = b.str[0], b.str[1], b.str[2]
# Create categorical columns using the descriptions
df["MDMAUD_recency"]   = b1.map(recency_map)
df["MDMAUD_frequency"] = b2.map(frequency_map)
df["MDMAUD_amount"]    = b3.map(amount_map)
df["MAJOR"] = df["MAJOR"].replace({" ": "Not a Major Donor", "X": "Major Donor"})
assert ((df["MAJOR"] == "Not a Major Donor") == df["MDMAUD"].str.upper().eq("XXXX")).all()
df.loc[df["MAJOR"] == "Not a Major Donor", ["MDMAUD_recency", "MDMAUD_frequency", "MDMAUD_amount"]] = "Not applicable"

df = df.drop(columns=["MDMAUD"]) # can be recovered by pipelines if needed via arithmetic interactions
# Resolve DOMAIN Code
s = df["DOMAIN"].replace(" ", np.nan).astype("string")
b = s.str.pad(2, side="right")
b1 = b.str[0]
b2 = b.str[1]
df["DOMAIN_urbanicity"] = b1.map(urbanicity_map)
df["DOMAIN_ses"] = b2.map(ses_map)
df = df.drop(columns=["DOMAIN"])
# Cluster fix nan
df["CLUSTER"] = df["CLUSTER"].replace(" ", np.nan) # Likely not Ordinal
# Age (has no - values like data dict is saying), so skip but resolve for ageflag
df["AGEFLAG"] = df["AGEFLAG"].replace({0: np.nan, "E": "Exact", "I": "Inferred from Date of Birth Field"})
df["HOMEOWNR"] = df["HOMEOWNR"].replace({" ": np.nan, "H": "Homeowner", "U": "Unknown"})
# Resolve child
child_map = {"B": "Both", "F": "Female", "M": "Male", " ": np.nan}
child_cols = ["CHILD03", "CHILD07", "CHILD12", "CHILD18"]
for col in child_cols:
    df[col] = df[col].replace(child_map)
df["GENDER"] = df["GENDER"].replace({"M": "Male", "F": "Female", "U": "Unknown", " ": np.nan, "J": "Joint Account, unknown gender"})
df["DATASRCE"] = df["DATASRCE"].replace({" ": np.nan, "1": "MetroMail", "2":"Polk", "3":"Both"})
# Not Ordinal/Int as 0 is equal to INF
df["SOLIH"] = df["SOLIH"].replace({" ": "can be mailed (Default)","00":"Do Not Solicit","01":"one solicitation per year","02":"two solicitations per year","03":"three solicitations per year","04":"four solicitations per year","05":"five solicitations per year","06":"six solicitations per year","12":"twelve solicitations per year"})
df["SOLP3"] = df["SOLP3"].replace({" ": "can be mailed (Default)","00":"Do Not Solicit or Mail","01":"one solicitation per year","02":"two solicitations per year","03":"three solicitations per year","04":"four solicitations per year","05":"five solicitations per year","06":"six solicitations per year","12":"twelve solicitations per year"})
df["GEOCODE"] = df["GEOCODE"].replace(" ", "No code has been assigned or did not match at any level")
df["LIFESRC"] = df["LIFESRC"].replace({" ": np.nan, "1": "MATCHED ON METRO MAIL ONLY","2":"MATCHED ON POLK ONLY","3":"MATCHED BOTH MM AND POLK"})
df["PEPSTRFL"] = df["PEPSTRFL"].replace({" ": np.nan, "X": "Has PEP Star RFA Status"})
df["GEOCODE2"] = df["GEOCODE2"].replace(" ", np.nan)
# Resolve interests
y_n_map_list = [
    "COLLECT1","VETERANS","BIBLE","CATLG","HOMEE","PETS","CDPLAY","STEREO",
    "PCOWNERS","PHOTO","CRAFTS","FISHER","GARDENIN","BOATS","WALKER",
    "KIDSTUFF","CARDS","PLATES"
]
for col in y_n_map_list:
    # Data dir says it is yes/no, but data is yes or missing, so I will go for missing as it is more general
    assert np.unique(df[col].dropna()).tolist() == [" ", "Y"]
    df[col] = df[col].replace({"Y": "Yes", " ": np.nan})

df = df.drop(columns=[
    "CONTROLN", # uninformative index column
    "RFA_2R", # constant col
    # Already decoded above and added semantics
    "MDMAUD_R", "MDMAUD_A", "MDMAUD_F",
    # Other target
    "TARGET_D",
])

# -- Dtypes
as_type_str = [
    "OSOURCE", "STATE", "ZIP", "Title",
    # RFA codes (not decoded as too many columns otherwise)
    "RFA_2","RFA_3","RFA_4","RFA_5","RFA_6","RFA_7","RFA_8","RFA_9",
    "RFA_10","RFA_11","RFA_12","RFA_13","RFA_14","RFA_15","RFA_16",
    "RFA_17","RFA_18","RFA_19","RFA_20","RFA_21","RFA_22","RFA_23","RFA_24"
]
as_type_cat = [
    "PVASTATE", "MAILCODE", "NOEXCH", "RECINHSE", "RECP3", "RECPGVG", "RECSWEEP",
    "MDMAUD_recency", "MDMAUD_frequency", "MDMAUD_amount", "MAJOR",
    "DOMAIN_urbanicity", "DOMAIN_ses", "CLUSTER", "CLUSTER2", "AGEFLAG", "HOMEOWNR",
    "CHILD03", "CHILD07", "CHILD12", "CHILD18", "GENDER", "DATASRCE", "SOLIH",
    "SOLP3", "GEOCODE", "GEOCODE2", "LIFESRC", "PEPSTRFL", "HPHONE_D",
    "RFA_2F", "RFA_2A",
    "TARGET_B", # will be dropped later, but making sure it is valid
] + y_n_map_list

for c in as_type_str:
    df[c] = df[c].replace(" ", np.nan)
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_type_cat] = df[as_type_cat].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [5]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 144,095
Columns: 478
Use sampling: False (sample size: 144,095)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['IC5', 'ZIP', 'POP901', 'AVGGIFT', 'POP903', 'POP902', 'HV2', 'HV1', 'RAMNTALL', 'IC2']
Rows remaining as candidates after top-10 filter: 792 (of 144,095)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [6]:
# Sample Rows
df_head

,ODATEDW,OSOURCE,STATE,ZIP,MAILCODE,PVASTATE,DOB,NOEXCH,RECINHSE,RECP3,RECPGVG,RECSWEEP,CLUSTER,AGE,AGEFLAG,HOMEOWNR,CHILD03,CHILD07,CHILD12,CHILD18,NUMCHLD,INCOME,GENDER,WEALTH1,HIT,MBCRAFT,MBGARDEN,MBBOOKS,MBCOLECT,MAGFAML,MAGFEM,MAGMALE,PUBGARDN,PUBCULIN,PUBHLTH,PUBDOITY,PUBNEWFN,PUBPHOTO,PUBOPP,DATASRCE,MALEMILI,MALEVET,VIETVETS,WWIIVETS,LOCALGOV,STATEGOV,FEDGOV,SOLP3,SOLIH,MAJOR,WEALTH2,GEOCODE,COLLECT1,VETERANS,BIBLE,CATLG,HOMEE,PETS,CDPLAY,STEREO,PCOWNERS,PHOTO,CRAFTS,FISHER,GARDENIN,BOATS,WALKER,KIDSTUFF,CARDS,PLATES,LIFESRC,PEPSTRFL,POP901,POP902,POP903,POP90C1,POP90C2,POP90C3,POP90C4,POP90C5,ETH1,ETH2,ETH3,ETH4,ETH5,ETH6,ETH7,ETH8,ETH9,ETH10,ETH11,ETH12,ETH13,ETH14,ETH15,ETH16,AGE901,AGE902,AGE903,AGE904,AGE905,AGE906,AGE907,CHIL1,CHIL2,CHIL3,AGEC1,AGEC2,AGEC3,AGEC4,AGEC5,AGEC6,AGEC7,CHILC1,CHILC2,CHILC3,CHILC4,CHILC5,HHAGE1,HHAGE2,HHAGE3,HHN1,HHN2,HHN3,HHN4,HHN5,HHN6,MARR1,MARR2,MARR3,MARR4,HHP1,HHP2,DW1,DW2,DW3,DW4,DW5,DW6,DW7,DW8,DW9,HV1,HV2,HV3,HV4,HU1,HU2,HU3,HU4,HU5,HHD1,HHD2,HHD3,HHD4,HHD5,HHD6,HHD7,HHD8,HHD9,HHD10,HHD11,HHD12,ETHC1,ETHC2,ETHC3,ETHC4,ETHC5,ETHC6,HVP1,HVP2,HVP3,HVP4,HVP5,HVP6,HUR1,HUR2,RHP1,RHP2,RHP3,RHP4,HUPA1,HUPA2,HUPA3,HUPA4,HUPA5,HUPA6,HUPA7,RP1,RP2,RP3,RP4,MSA,ADI,DMA,IC1,IC2,IC3,IC4,IC5,IC6,IC7,IC8,IC9,IC10,IC11,IC12,IC13,IC14,IC15,IC16,IC17,IC18,IC19,IC20,IC21,IC22,IC23,HHAS1,HHAS2,HHAS3,HHAS4,MC1,MC2,MC3,TPE1,TPE2,TPE3,TPE4,TPE5,TPE6,TPE7,TPE8,TPE9,PEC1,PEC2,TPE10,TPE11,TPE12,TPE13,LFC1,LFC2,LFC3,LFC4,LFC5,LFC6,LFC7,LFC8,LFC9,LFC10,OCC1,OCC2,OCC3,OCC4,OCC5,OCC6,OCC7,OCC8,OCC9,OCC10,OCC11,OCC12,OCC13,EIC1,EIC2,EIC3,EIC4,EIC5,EIC6,EIC7,EIC8,EIC9,EIC10,EIC11,EIC12,EIC13,EIC14,EIC15,EIC16,OEDC1,OEDC2,OEDC3,OEDC4,OEDC5,OEDC6,OEDC7,EC1,EC2,EC3,EC4,EC5,EC6,EC7,EC8,SEC1,SEC2,SEC3,SEC4,SEC5,AFC1,AFC2,AFC3,AFC4,AFC5,AFC6,VC1,VC2,VC3,VC4,ANC1,ANC2,ANC3,ANC4,ANC5,ANC6,ANC7,ANC8,ANC9,ANC10,ANC11,ANC12,ANC13,ANC14,ANC15,POBC1,POBC2,LSC1,LSC2,LSC3,LSC4,VOC1,VOC2,VOC3,HC1,HC2,HC3,HC4,HC5,HC6,HC7,HC8,HC9,HC10,HC11,HC12,HC13,HC14,HC15,HC16,HC17,HC18,HC19,HC20,HC21,MHUC1,MHUC2,AC1,AC2,ADATE_2,ADATE_3,ADATE_4,ADATE_5,ADATE_6,ADATE_7,ADATE_8,ADATE_9,ADATE_10,ADATE_11,ADATE_12,ADATE_13,ADATE_14,ADATE_15,ADATE_16,ADATE_17,ADATE_18,ADATE_19,ADATE_20,ADATE_21,ADATE_22,ADATE_23,ADATE_24,RFA_2,RFA_3,RFA_4,RFA_5,RFA_6,RFA_7,RFA_8,RFA_9,RFA_10,RFA_11,RFA_12,RFA_13,RFA_14,RFA_15,RFA_16,RFA_17,RFA_18,RFA_19,RFA_20,RFA_21,RFA_22,RFA_23,RFA_24,CARDPROM,MAXADATE,NUMPROM,CARDPM12,NUMPRM12,RDATE_3,RDATE_4,RDATE_5,RDATE_6,RDATE_7,RDATE_8,RDATE_9,RDATE_10,RDATE_11,RDATE_12,RDATE_13,RDATE_14,RDATE_15,RDATE_16,RDATE_17,RDATE_18,RDATE_19,RDATE_20,RDATE_21,RDATE_22,RDATE_23,RDATE_24,RAMNT_3,RAMNT_4,RAMNT_5,RAMNT_6,RAMNT_7,RAMNT_8,RAMNT_9,RAMNT_10,RAMNT_11,RAMNT_12,RAMNT_13,RAMNT_14,RAMNT_15,RAMNT_16,RAMNT_17,RAMNT_18,RAMNT_19,RAMNT_20,RAMNT_21,RAMNT_22,RAMNT_23,RAMNT_24,RAMNTALL,NGIFTALL,CARDGIFT,MINRAMNT,MINRDATE,MAXRAMNT,MAXRDATE,LASTGIFT,LASTDATE,FISTDATE,NEXTDATE,TIMELAG,AVGGIFT,TARGET_B,HPHONE_D,RFA_2F,RFA_2A,CLUSTER2,GEOCODE2,Title,MDMAUD_recency,MDMAUD_frequency,MDMAUD_amount,DOMAIN_urbanicity,DOMAIN_ses
0,1995-01-01,MM1,CA,91702,Address is OK,NaN,1916-01-01,0-nan-case,Not an In House Record,Not a P3 Record,Not a Planned Giving Record,Not a Sweepstakes Record,21,82.0,Exact,Homeowner,NaN,NaN,NaN,NaN,NaN,1.0,Male,1.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Both,0,24,25,30,3,1,1,can be mailed (Default),can be mailed (Default),Not a Major Donor,NaN,No code has been assigned or did not match at any level,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1510,344,464,99,0,0,49,51,59,2,1,2,67,0,0,1,1,0,0,0,60,0,1,5,27,35,40,30,41,45,32,44,38,18,19,30,16,11,11,8,5,19,17,33,18,13,23,9,20,19,27,53,37,24,13,53,12,6,29,220,317,74,67,5,26,20,12,0,0,0,1667,1761,5,5,49,51,95,5,0,44,74,54,33,85,15,11,3,8,14,20,6,15,32,11,1,2,0,23,67,88,97,99,5,16,28,46,44,16,7,17,9,0,26,13,12,0,43,64,89,97,4480.0,13.0,803.0,224,250,295,316,8745,25,27,17,19,5,4,2,0,0,18,32,18,22,2,5,2,0,0,24,14,16,16,44,56

In [7]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,PLATES,category,143127.0,99.33,1.0,Yes
1,HOMEE,category,142525.0,98.91,1.0,Yes
2,CARDS,category,142376.0,98.81,1.0,Yes
3,CHILD03,category,142045.0,98.58,3.0,"Male, Female, Both"
4,PVASTATE,category,142028.0,98.57,1.0,P
5,KIDSTUFF,category,141519.0,98.21,1.0,Yes
6,CHILD07,category,141179.0,97.98,3.0,"Male, Female, Both"
7,CHILD12,category,140711.0,97.65,3.0,"Male, Female, Both"
8,BOATS,category,140572.0,97.56,1.0,Yes
9,CHILD18,category,138863.0,96.37,3.0,"Male, Female, Both"


In [8]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
AGE,144091.0,61.628853,16.665193,1.000000,98.0
NUMCHLD,22728.0,1.533879,0.811377,1.000000,7.0
INCOME,127618.0,3.960241,1.843408,1.000000,7.0
WEALTH1,88574.0,5.417143,2.734297,0.000000,9.0
HIT,144095.0,4.070211,9.790032,0.000000,242.0
MBCRAFT,76918.0,0.156348,0.478916,0.000000,6.0
MBGARDEN,76918.0,0.060467,0.264614,0.000000,4.0
MBBOOKS,76918.0,1.155022,1.702199,0.000000,9.0
MBCOLECT,76819.0,0.065635,0.300941,0.000000,6.0
MAGFAML,76918.0,0.461218,0.829841,0.000000,9.0


In [9]:
# Categorical Feature Statistics
cat_stats

value  \
column            rank                                                            
ADATE_10          1                                         1995-10-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1995-11-01 00:00:00   
ADATE_11          1                                         1995-10-01 00:00:00   
                  2                                         1995-09-01 00:00:00   
                  3                                                        <NA>   
                  4                                         1995-11-01 00:00:00   
                  5                                         1995-08-01 00:00:00   
ADATE_12          1                                         1995-08-01 00:00:00   
                  2                                         1995-09-01 00:00:00   
                  3                                                        <NA>   
                  4                                         1995-10-01 00:00:00   
                  5                                         1995-07-01 00:00:00   
ADATE_13          1                                         1995-07-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1995-02-01 00:00:00   
                  4                                         1995-06-01 00:00:00   
ADATE_14          1                                         1995-06-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1995-04-01 00:00:00   
ADATE_15          1                                                        <NA>   
                  2                                         1995-04-01 00:00:00   
ADATE_16          1                                         1995-03-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1995-04-01 00:00:00   
                  4                                         1995-02-01 00:00:00   
ADATE_17          1                                         1995-02-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1995-01-01 00:00:00   
                  4                                         1995-03-01 00:00:00   
ADATE_18          1                                         1995-01-01 00:00:00   
                  2                                         1994-12-01 00:00:00   
                  3                                                        <NA>   
                  4                                         1994-11-01 00:00:00   
                  5                                         1994-09-01 00:00:00   
ADATE_19          1                                         1994-11-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1994-09-01 00:00:00   
                  4                                         1994-10-01 00:00:00   
ADATE_2           1                                         1997-06-01 00:00:00   
                  2                                         1997-04-01 00:00:00   
ADATE_20          1                                                        <NA>   
                  2                                         1994-11-01 00:00:00   
                  3                                         1994-12-01 00:00:00   
ADATE_21          1                                         1994-10-01 00:00:00   
                  2                                                        <NA>   
                  3                                         1994-09-01 00:00:00   
ADATE_2

In [10]:
# Target Distribution
target_df

,count,pct
TARGET_B,,
0,136639,94.83
1,7456,5.17


## Task Curation

In [11]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [12]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [13]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to pva_revenue_prediction_kddcup98/019db49b-9c05-72b8-b4a5-7b967a18225b
019db49b-9c05-72b8-b4a5-7b967a18225b
ac7bab79289e3eb4b1139306571d44ec21ba05681e7582f10c38bf344482816b
